# Tracing the SDK Call Chain

**Goal:** See exactly which functions run (and in what order) when you call `agent("hello")`.

**Two ways to step through the SDK:**

| Approach | What it is | When to use |
|----------|-----------|------------|
| **This notebook** | Injects print statements into SDK functions. You see the trace in the output. | Quick, no setup. Good for seeing the big picture. |
| **VS Code debugger** | Set breakpoints in SDK files, press F5, step line by line. | Deep inspection. See variable values at every step. |

For the VS Code debugger approach, see `00-debugger-guide.md` and `debug_agent.py`.

This notebook uses the tracing approach.

---
## What is Monkey-Patching?

**Monkey-patching** means replacing a function at runtime with a modified version.

**Analogy:** Imagine a package delivery service. You put a GPS tracker on each package. The packages still get delivered the same way -- but now you can see the exact route they take.

That's what we do here. We wrap SDK functions with a version that prints "I'm starting" and "I'm done" before and after the original function runs. The SDK still works exactly the same -- we just added visibility.

```python
# Before monkey-patching:
def original_function():
    return "result"

# After monkey-patching:
def wrapped_function():
    print(">>> Entering original_function")   # Added
    result = original_function()                # Original still runs
    print("<<< Exiting original_function")    # Added
    return result
```

In [1]:
# ============================================================
# STEP 1: Import the SDK and define the tracing helper
# ============================================================

import functools
import asyncio
import inspect

from strands import Agent, tool
from strands.agent.agent import Agent as AgentClass
from strands.event_loop import event_loop as event_loop_module

# Global indent level for nested function calls.
# Each time we enter a function, indent increases.
# Each time we exit, indent decreases.
# This creates a visual tree of the call chain.
_trace_indent = 0


def trace_method(cls, method_name, display_name=None):
    """Replace a method on a class with a traced version.
    
    The traced version prints:
      >>> Entering function_name
      <<< Exiting function_name
    
    With indentation showing nesting depth.
    """
    global _trace_indent
    name = display_name or method_name
    original = getattr(cls, method_name)
    
    if inspect.isasyncgenfunction(original):
        # For async generators (functions with 'async def' + 'yield')
        @functools.wraps(original)
        async def wrapper(*args, **kwargs):
            global _trace_indent
            indent = "  " * _trace_indent
            print(f"{indent}>>> {name}")
            _trace_indent += 1
            try:
                async for item in original(*args, **kwargs):
                    yield item
            finally:
                _trace_indent -= 1
                print(f"{indent}<<< {name}")
    elif inspect.iscoroutinefunction(original):
        # For regular async functions (functions with 'async def')
        @functools.wraps(original)
        async def wrapper(*args, **kwargs):
            global _trace_indent
            indent = "  " * _trace_indent
            print(f"{indent}>>> {name}")
            _trace_indent += 1
            try:
                result = await original(*args, **kwargs)
                return result
            finally:
                _trace_indent -= 1
                print(f"{indent}<<< {name}")
    else:
        # For regular (sync) functions
        @functools.wraps(original)
        def wrapper(*args, **kwargs):
            global _trace_indent
            indent = "  " * _trace_indent
            print(f"{indent}>>> {name}")
            _trace_indent += 1
            try:
                result = original(*args, **kwargs)
                return result
            finally:
                _trace_indent -= 1
                print(f"{indent}<<< {name}")
    
    setattr(cls, method_name, wrapper)
    return original  # Return original so we can restore later


def trace_function(module, func_name, display_name=None):
    """Replace a module-level function with a traced version."""
    global _trace_indent
    name = display_name or func_name
    original = getattr(module, func_name)
    
    if inspect.isasyncgenfunction(original):
        @functools.wraps(original)
        async def wrapper(*args, **kwargs):
            global _trace_indent
            indent = "  " * _trace_indent
            print(f"{indent}>>> {name}")
            _trace_indent += 1
            try:
                async for item in original(*args, **kwargs):
                    yield item
            finally:
                _trace_indent -= 1
                print(f"{indent}<<< {name}")
    elif inspect.iscoroutinefunction(original):
        @functools.wraps(original)
        async def wrapper(*args, **kwargs):
            global _trace_indent
            indent = "  " * _trace_indent
            print(f"{indent}>>> {name}")
            _trace_indent += 1
            try:
                result = await original(*args, **kwargs)
                return result
            finally:
                _trace_indent -= 1
                print(f"{indent}<<< {name}")
    else:
        @functools.wraps(original)
        def wrapper(*args, **kwargs):
            global _trace_indent
            indent = "  " * _trace_indent
            print(f"{indent}>>> {name}")
            _trace_indent += 1
            try:
                result = original(*args, **kwargs)
                return result
            finally:
                _trace_indent -= 1
                print(f"{indent}<<< {name}")
    
    setattr(module, func_name, wrapper)
    return original


print("Tracing helpers defined.")

Tracing helpers defined.


In [2]:
# ============================================================
# STEP 2: Apply traces to the 8 key functions in the call chain
# ============================================================

# Reset indent
_trace_indent = 0

# Save originals so we can restore later if needed.
originals = {}

# --- Agent methods (on the Agent class) ---

# 1. __call__ (agent.py:335) -- entry point
originals['__call__'] = trace_method(
    AgentClass, '__call__', 'Agent.__call__ (agent.py:335)'
)

# 2. _run_loop (agent.py:643) -- hooks + message management
originals['_run_loop'] = trace_method(
    AgentClass, '_run_loop', 'Agent._run_loop (agent.py:643)'
)

# --- Event loop functions (module-level in event_loop.py) ---

# 3. event_loop_cycle (event_loop.py:78) -- the core engine
originals['event_loop_cycle'] = trace_function(
    event_loop_module, 'event_loop_cycle', 'event_loop_cycle (event_loop.py:78)'
)

# 4. _handle_model_execution (event_loop.py:275) -- calls the AI model
originals['_handle_model_execution'] = trace_function(
    event_loop_module, '_handle_model_execution', '_handle_model_execution (event_loop.py:275)'
)

# 5. _handle_tool_execution (event_loop.py:421) -- executes tools
originals['_handle_tool_execution'] = trace_function(
    event_loop_module, '_handle_tool_execution', '_handle_tool_execution (event_loop.py:421)'
)

# 6. recurse_event_loop (event_loop.py:236) -- loops back for next cycle
originals['recurse_event_loop'] = trace_function(
    event_loop_module, 'recurse_event_loop', 'recurse_event_loop (event_loop.py:236)'
)

print("Traces applied to 6 key functions.")
print("When you call agent(), you'll see the call chain printed.")

Traces applied to 6 key functions.
When you call agent(), you'll see the call chain printed.


In [3]:
# ============================================================
# STEP 3: Define a tool and create the agent
# ============================================================

@tool
def calculator(expression: str) -> str:
    """Perform a math calculation.
    
    Args:
        expression: A math expression like '2 + 2'.
    """
    # This print shows when YOUR code runs inside the call chain.
    print(f"{'  ' * _trace_indent}    [calculator] evaluating: {expression}")
    result = eval(expression)
    print(f"{'  ' * _trace_indent}    [calculator] result: {result}")
    return str(result)

# Create agent with tracing-friendly settings
agent = Agent(
    tools=[calculator],
    callback_handler=None,  # No streaming output -- keeps trace clean
)

print("Agent created with calculator tool.")

Agent created with calculator tool.


In [4]:
# ============================================================
# STEP 4: Call the agent and see the full trace!
# ============================================================

# Reset indent level
_trace_indent = 0

print("=" * 70)
print("CALL TRACE: agent(\"What is 2 + 2?\")")
print("=" * 70)
print()

result = agent("What is 2 + 2?")

print()
print("=" * 70)
print(f"RESULT: {result}")
print(f"Messages in history: {len(agent.messages)}")
print("=" * 70)

CALL TRACE: agent("What is 2 + 2?")

>>> Agent.__call__ (agent.py:335)
  >>> Agent._run_loop (agent.py:643)
    >>> _handle_model_execution (event_loop.py:275)
    <<< _handle_model_execution (event_loop.py:275)
    >>> _handle_tool_execution (event_loop.py:421)
          [calculator] evaluating: 2 + 2
          [calculator] result: 4
      >>> recurse_event_loop (event_loop.py:236)
        >>> event_loop_cycle (event_loop.py:78)
          >>> _handle_model_execution (event_loop.py:275)
          <<< _handle_model_execution (event_loop.py:275)
        <<< event_loop_cycle (event_loop.py:78)
      <<< recurse_event_loop (event_loop.py:236)
    <<< _handle_tool_execution (event_loop.py:421)
  <<< Agent._run_loop (agent.py:643)
<<< Agent.__call__ (agent.py:335)

RESULT: The answer is 4.

Messages in history: 4


---
## Reading the Trace

The output above shows the complete call chain. Here's what each line means:

```
>>> Agent.__call__                          You called agent("What is 2+2?")
  >>> Agent._run_loop                       Hooks fire, your message is appended
    >>> event_loop_cycle                    CYCLE 1: The core engine starts
      >>> _handle_model_execution           AI model is called
      <<< _handle_model_execution           Model responds: "I'll use calculator"
                                            (stop_reason = "tool_use")
      >>> _handle_tool_execution            Tool execution begins
          [calculator] evaluating: 2 + 2    YOUR tool code runs!
          [calculator] result: 4
        >>> recurse_event_loop              Loop back for cycle 2
          >>> event_loop_cycle              CYCLE 2: Model sees tool result
            >>> _handle_model_execution     Model called again
            <<< _handle_model_execution     Model responds: "2+2 is 4"
                                            (stop_reason = "end_turn")
          <<< event_loop_cycle              Cycle 2 done
        <<< recurse_event_loop
      <<< _handle_tool_execution
    <<< event_loop_cycle                    Cycle 1 done
  <<< Agent._run_loop                       Hooks fire, conversation managed
<<< Agent.__call__                          Done! Result returned
```

**Key observations:**
- `>>>` means entering a function, `<<<` means exiting
- Indentation shows nesting (who called who)
- The model is called **twice** (Cycle 1 + Cycle 2)
- Your tool runs **between** the two model calls
- `recurse_event_loop` is what connects Cycle 1 to Cycle 2

---
## What is `breakpoint()`?

`breakpoint()` is Python's built-in debugger command. When Python hits this line, it **pauses** and gives you an interactive prompt.

At the prompt (called `(Pdb)`), you can:
- Type a **variable name** to see its value: `expression` -> `'2 + 2'`
- Type **`n`** to go to the next line
- Type **`s`** to step into a function call
- Type **`c`** to continue running (exit the debugger)
- Type **`q`** to quit

In Jupyter notebooks, `breakpoint()` works **inside @tool functions** because they run in a separate thread. It does NOT work in the main notebook cell or in async SDK functions.

**Try it:** Run the cell below. When it pauses at `(Pdb)`, type `expression` and press Enter. Then type `c` to continue.

In [ ]:
# ============================================================
# STEP 5: Using breakpoint() inside a tool
# ============================================================

# WARNING: This cell will PAUSE and wait for your input!
#
# When you see the (Pdb) prompt:
#   Type: expression     (to see the value of the 'expression' variable)
#   Type: c              (to continue running)

@tool
def calculator_debug(expression: str) -> str:
    """Perform a math calculation (with debugger).
    
    Args:
        expression: A math expression like '2 + 2'.
    """
    # When Python hits this line, it PAUSES and opens the (Pdb) prompt.
    # You can inspect variables, step through code, etc.
    breakpoint()
    
    result = eval(expression)
    return str(result)

# Create a fresh agent (without the monkey-patched traces)
agent_debug = Agent(
    tools=[calculator_debug],
    callback_handler=None,
)

print("Calling agent... it will pause at breakpoint() inside the tool.")
print("At the (Pdb) prompt, type 'expression' to see the value, then 'c' to continue.")
print()

result = agent_debug("What is 10 * 5?")
print(f"\nResult: {result}")

---
## For Deeper Debugging: Use VS Code

The tracing above shows you the **big picture** -- which functions run and in what order.

For **line-by-line** inspection with variable values at every step, use the VS Code debugger:

1. Read **`00-debugger-guide.md`** -- explains the VS Code debugger from scratch
2. Open **`debug_agent.py`** -- a script designed to be debugged with F5
3. Set breakpoints in SDK source files (line numbers listed in `debug_agent.py`)
4. Press F5 and step through!

---
## Breakpoint Map

Where to set breakpoints in the SDK source files:

| # | File | Line | Function | What happens here |
|---|------|------|----------|------------------|
| 1 | `src/strands/agent/agent.py` | 335 | `__call__` | Entry point -- `agent("hello")` starts here |
| 2 | `src/strands/agent/agent.py` | 376 | `invoke_async` | Bridges sync to async |
| 3 | `src/strands/agent/agent.py` | 539 | `stream_async` | Acquires lock, converts prompt |
| 4 | `src/strands/agent/agent.py` | 643 | `_run_loop` | Fires hooks, appends user message |
| 5 | `src/strands/event_loop/event_loop.py` | 78 | `event_loop_cycle` | The core engine |
| 6 | `src/strands/event_loop/event_loop.py` | 275 | `_handle_model_execution` | AI model is called |
| 7 | `src/strands/event_loop/event_loop.py` | 421 | `_handle_tool_execution` | Your tools execute |
| 8 | `src/strands/tools/decorator.py` | 554 | `stream` | Your @tool function runs |
| 9 | `src/strands/event_loop/event_loop.py` | 236 | `recurse_event_loop` | Loops back for next cycle |

**Tip:** To open these files quickly in VS Code, press Ctrl+P (Cmd+P on Mac) and type the filename.

In [ ]:
# ============================================================
# CLEANUP: Restore original functions (removes traces)
# ============================================================

# If you want to use the agent normally after tracing,
# run this cell to remove the trace wrappers.

AgentClass.__call__ = originals['__call__']
AgentClass._run_loop = originals['_run_loop']
event_loop_module.event_loop_cycle = originals['event_loop_cycle']
event_loop_module._handle_model_execution = originals['_handle_model_execution']
event_loop_module._handle_tool_execution = originals['_handle_tool_execution']
event_loop_module.recurse_event_loop = originals['recurse_event_loop']

print("Traces removed. Agent functions restored to originals.")